# 02 · Local TimesFM 2.5 forecast demo

Zero-shot TimesFM 2.5 forecast over the configured horizon, built through
`geaptimes.models.factory.ForecastFactory`, plotted against the held-out **TEST** actuals.

- Reads the prepped table from BigQuery and downloads the TimesFM checkpoint on first use
  (needs ADC auth + network).
- Univariate (covariates are opt-in/deferred). Quantile band uses the standardized
  `QUANTILES = [0.1, 0.3, 0.5, 0.7, 0.9]`.
- Run with the **geapTimes (uv 3.11)** kernel.

### Colab (optional)

To run in Colab instead of the local `uv` kernel, uncomment the cell below to install the
package and authenticate. Keep it commented for the local flow.

In [ ]:
# !pip install -q -e ..
# from google.colab import auth; auth.authenticate_user()

In [ ]:
# Load env vars (GCP_PROJECT) from the repo-root .env so config ${VAR} substitution resolves.
# The CLI sets these via `GCP_PROJECT=… uv run …`; the kernel needs them loaded explicitly.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())  # walks up from data_notebooks/ to find ../.env

In [ ]:
from geaptimes.models.base import QUANTILE_COLUMNS
from geaptimes.models.factory import ForecastFactory
from geaptimes.naming import table_names
from geaptimes.schemas import ExperimentConfig

cfg = ExperimentConfig.from_yaml("../config/base_config.yaml")
model_cfg = next(m for m in cfg.models if m.enabled and m.params.type == "timesfm")
model_cfg.name, cfg.forecast.horizon

In [ ]:
# Build the forecaster and run a zero-shot forecast (fit() is a no-op for TimesFM).
forecaster = ForecastFactory.create(model_cfg, cfg)
forecaster.fit()
result = forecaster.predict()
print(result.metadata)
result.predictions.head(10)

In [ ]:
# Fetch the held-out TEST actuals from the prepped table for comparison.
from google.cloud import bigquery

prepped = table_names(cfg)["prepped"]
prepped_id = f"{cfg.project.id}.{cfg.project.bq_dataset}.{prepped}"
data = cfg.data
sql = (
    f"SELECT {data.series_column} AS series, {data.time_column} AS date, "
    f"{data.target_column} AS actual "
    f"FROM `{prepped_id}` WHERE splits = 'TEST'"
)
actuals = bigquery.Client(project=cfg.project.id).query(sql).to_dataframe()
actuals.head()

In [ ]:
# Forecast vs TEST actuals — per-station grid: Full View (left) | Zoomed last 30d (right).
# Combines statmike's full/zoom split with nested quantile bands + actuals overlay + per-station error.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

N_STATIONS = 3  # top stations by mean TEST demand

top_stations = (
    actuals.groupby("series")["actual"].mean().sort_values(ascending=False).head(N_STATIONS).index.tolist()
)

# Pull full pre-TEST history for the selected stations in one query.
hist_sql = (
    f"SELECT {data.series_column} AS series, {data.time_column} AS date, "
    f"{data.target_column} AS actual "
    f"FROM `{prepped_id}` "
    f"WHERE {data.series_column} IN UNNEST(@stations) AND splits != 'TEST' "
    f"ORDER BY series, date"
)
hist_job = bigquery.QueryJobConfig(
    query_parameters=[bigquery.ArrayQueryParameter("stations", "STRING", top_stations)]
)
hist_all = bigquery.Client(project=cfg.project.id).query(hist_sql, job_config=hist_job).to_dataframe()


def draw(ax, hist_s, act_s, pred_s, *, zoom=False):
    ax.plot(hist_s["date"], hist_s["actual"], color="steelblue", lw=1.2, label="history")
    ax.plot(act_s["date"], act_s["actual"], "o-", color="black", lw=1.3, ms=4, label="actual (TEST)")
    ax.plot(pred_s["date"], pred_s["forecast"], "--", color="darkorange", lw=1.5, label="forecast")
    ax.fill_between(pred_s["date"], pred_s["q10"], pred_s["q90"],
                    color="sandybrown", alpha=0.35, label="q10-q90")
    ax.fill_between(pred_s["date"], pred_s["q30"], pred_s["q70"],
                    color="sandybrown", alpha=0.60, label="q30-q70")
    ax.grid(True, linestyle=":")
    if zoom:  # last 30d of history through end of forecast
        ax.set_xlim(hist_s["date"].max() - pd.Timedelta(days=30),
                    pred_s["date"].max() + pd.Timedelta(days=1))


fig, axes = plt.subplots(N_STATIONS, 2, figsize=(16, 4 * N_STATIONS), squeeze=False)
for i, name in enumerate(top_stations):
    hist_s = hist_all[hist_all["series"] == name].sort_values("date")
    act_s = actuals[actuals["series"] == name].sort_values("date")
    pred_s = result.predictions[result.predictions["series"] == name].sort_values("date")

    merged = act_s.merge(pred_s[["date", "forecast"]], on="date", how="inner")
    mae = (merged["actual"] - merged["forecast"]).abs().mean()
    rmse = np.sqrt(((merged["actual"] - merged["forecast"]) ** 2).mean())

    draw(axes[i, 0], hist_s, act_s, pred_s, zoom=False)
    draw(axes[i, 1], hist_s, act_s, pred_s, zoom=True)
    axes[i, 0].set_title(f"Full — {name}   (MAE {mae:.1f} · RMSE {rmse:.1f})")
    axes[i, 1].set_title(f"Zoom (last 30d) — {name}")
    axes[i, 0].set_ylabel(cfg.data.target_column)

axes[0, 0].legend(loc="upper left", fontsize=8, ncol=2)
fig.suptitle("TimesFM 2.5 — forecast vs TEST actuals", fontsize=13, y=1.0)
plt.tight_layout()
plt.show()

print("quantile columns:", QUANTILE_COLUMNS)